# Week 2 Assignment 1 — ML Task Framing

## 1) My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring.

**ML task type:** Scoring (supervised opportunity scoring using a binary proxy label).

**Research question:** Which content pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

I chose scoring because the practical output is a prioritized review queue: each page receives an opportunity/risk score so a human reviewer can decide which pages deserve attention first. The starter workflow uses observable page-level signals and a binary decline proxy rather than a product decision.


## 2) Target or proxy

For this starter assignment, I will use **`is_declining_proxy`** as the target/proxy.

The proxy is defined as:

`trend_direction == "down"`

This is a **current-window proxy**, not a future causal outcome. It is useful for learning the workflow, but a stronger future version would define a clear decision point and predict a later outcome, such as decline or recovery in the next 30 days.

The target supports a review score: pages with stronger evidence of decline/opportunity can be ranked higher for human review.


## 3) Success metric

**Primary success metric: Precision@50.**

Precision@50 answers: of the 50 pages the scoring system ranks highest for review, how many are actually positive according to the chosen proxy label?

This metric matches the real action because reviewer capacity is limited. I care about making the top of the review queue useful, rather than only optimizing an overall classification score.


## 4) The unit of analysis, as a real dataframe

**Unit of analysis:** one row = one unique pseudonymized content page (`content_id`).

The starter feature-preparation code keeps pages with impressions and sufficient age, then deduplicates by `content_id`. I will load the starter CSV and create the proxy target so the unit of analysis is visible in an actual dataframe.


In [ ]:
from pathlib import Path
import pandas as pd

candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv. "
        "Run this notebook from the repository or adjust DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

# Match the starter preparation conditions.
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Starter proxy target.
df["is_declining_proxy"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

display(
    df[
        [
            "content_id",
            "client_id",
            "impressions_90d",
            "sessions_90d",
            "content_age_days",
            "trend_direction",
            "is_declining_proxy",
        ]
    ].head(10)
)


In [ ]:
print(f"Rows after starter filtering/deduplication: {len(df):,}")
print(f"Unique content_id values: {df['content_id'].nunique():,}")
print("Target counts:")

display(
    df["is_declining_proxy"]
    .value_counts()
    .rename(index={0: "not declining proxy", 1: "declining proxy"})
)


### Target column sketch

| trend_direction | is_declining_proxy |
|---|---:|
| `down` | 1 |
| anything else | 0 |

The code above creates this target from the observed `trend_direction` field. I am calling it a **proxy** because the repository guide notes that this starter label is calculated from the current window rather than a future outcome.


## 5) Why ML beats a fixed rule here

A fixed rule can flag simple cases, such as pages that are old, have high impressions, or show a downward trend. However, the review decision depends on several signals together: visibility, freshness, position, content depth, clicks/sessions, CTR, engagement, and other observed measurements.

A learned scoring model can combine these signals and interactions into a continuous priority score instead of relying on one manually chosen threshold. That makes it useful for ranking the limited set of pages a human should review first.

I would still keep a transparent rule-based baseline. The ML model should earn its place by improving a decision-focused metric such as Precision@50, and the final output should remain decision support rather than a claim that refreshing a page will definitely cause recovery.


## 6) Self-check

- [x] Task type is named and justified.
- [x] Target/proxy is defined and its limitation is stated.
- [x] Success metric is defined and connected to reviewer capacity.
- [x] Unit of analysis is one unique content page and is shown as a dataframe.
- [x] The action supported by the score is a ranked human review queue.
- [x] The ML-vs-rule explanation uses observable signals and decision-support framing.
- [ ] Notebook executed top-to-bottom without errors — **run all cells before committing.**
- [ ] Notebook committed at `work/notebooks/w02_ml_task_framing.ipynb` — **complete this in GitHub.**
